# Описание источников

Проверка способов получения и качества сохранения новостей. Окно публикаций: **26 октября — 8 ноября 2026**, Москва; конец 9 ноября исключён. Это обзор сбора, основной EDA новостей — следующий этап.

Каждая из 34 строк исходного Excel имеет результат. Сайты клубов и источники только структурированных данных исключены по решению пользователя. Наличие RSS не доказывает полноту архива; HTTP 200 не доказывает наличие текста.

Даты публикации заявлены источниками. Текущая редакция статьи могла измениться позже; `first_seen_at` относится к нашему сбору сейчас, не к моменту первого появления в интернете.

In [1]:
from pathlib import Path
import json, html
import pandas as pd
from IPython.display import display, HTML
ROOT = Path.cwd()
if ROOT.name == 'EDA':
    ROOT = ROOT.parent
assert (ROOT / 'data').is_dir(), 'Запустите ноутбук из корня проекта или EDA/'
SHOW_FULL_TEXT = False  # True: локально показать полные тексты из news.csv
audit = pd.read_csv(ROOT / 'EDA' / 'source_audit.csv').fillna('')
display(audit[['source','status','news','partial','review','posts','rerun_passed']])

,source,status,news,partial,review,posts,rerun_passed
0,Sky Sports,Извлечение статей проверено,123,0,129,0,True
1,ESPN,Извлечение статей проверено,22,0,37,0,True
2,talkSPORT,Рабочий маршрут не найден,0,0,0,0,False
3,The Independent,Извлечение статей проверено,148,3,79,0,True
4,Fantasy Football Scout,Извлечение статей проверено,88,0,39,0,True
5,RotoWire,Извлечение статей проверено,19,2,17,0,True
6,Sports Mole,Извлечение статей проверено,562,0,442,0,True
7,FPL,Исключён по решению пользователя,0,0,0,0,False
8,Официальные сайты 20 клубов,Исключён по решению пользователя,0,0,0,0,False
9,Football London,Извлечение статей проверено,289,0,15,0,True


## Что и как проверено

RSS/Atom — быстрый поиск новых URL; публичный API — даты, пагинация и текст, если доступен; HTML — извлечение статьи; карты сайта и архивы — поиск истории. У каждого способа есть отдельные результаты в Excel. Пустое окно, запрет доступа, ошибка парсера и отсутствие английских статей — разные причины. Платные API не подключались.

In [2]:
display(audit[['source','rss','api','html','archive_limits','reason']])
routes = pd.read_json(ROOT / 'EDA' / 'source_routes.json')
display(routes.groupby(['kind','status'], dropna=False).size().rename('проверок').reset_index())

,source,rss,api,html,archive_limits,reason
0,Sky Sports,https://www.skysports.com/rss/11095,Не подтверждён в проведённой проверке,Проверено извлечение текста,Достигнут бюджет страниц архива Sky,Извлечение статей проверено. Полнота архива не...
1,ESPN,https://www.espn.com/espn/rss/soccer/news,Не подтверждён в проведённой проверке,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...
2,talkSPORT,,Не подтверждён в проведённой проверке,Не подтверждено,,robots.txt запрещает обращения сборщика к пров...
3,The Independent,https://www.independent.co.uk/sport/football/p...,Не подтверждён в проведённой проверке,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...
4,Fantasy Football Scout,https://www.fantasyfootballscout.co.uk/feed,https://www.fantasyfootballscout.co.uk/wp-json...,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...
5,RotoWire,https://www.rotowire.com/rss/news.php?sport=SO...,Не подтверждён в проведённой проверке,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...
6,Sports Mole,https://www.sportsmole.co.uk/rss.xml\nhttps://...,Не подтверждён в проведённой проверке,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...
7,FPL,,Не подтверждён в проведённой проверке,Не подтверждено,,Источник структурированных данных исключён: се...
8,Официальные сайты 20 клубов,,Не подтверждён в проведённой проверке,Не подтверждено,,Официальные сайты клубов исключены по решению ...
9,Football London,https://www.football.london/?service=rss\nhttp...,Не подтверждён в проведённой проверке,Проверено извлечение текста,,Извлечение статей проверено. Полнота архива не...


,kind,status,проверок
0,article_discovery_alternative,200.0,2
1,html_alternative,200.0,2
2,html_alternative,404.0,1
3,html_listing,200.0,23
4,html_listing,403.0,1
5,html_listing,NaN,4
6,listing_json_html_fragment,200.0,1
7,public_news_api,200.0,3
8,public_news_api,403.0,1
9,public_undocumented_news_api,NaN,1


## Сохранённые наборы

`news.csv` — статьи, прошедшие текущий фильтр АПЛ; `posts.csv` — отдельный формат коротких постов; `partial.csv` — неполные тексты; `review.csv` — неопределённые случаи; `excluded.csv` — исключённые форматы/языки. Одна строка — публикация. CSV хранит полные тексты локально, вложенные списки представлены JSON-строками. Повторный запуск добавляет новые URL; уже сохранённые строки не теряются. Для обновления текста существующей статьи используется `--refresh-existing`; прежняя редакция сохраняется в revisions.jsonl.

In [3]:
frames = {}
for folder in sorted((ROOT / 'data').iterdir()):
    if not folder.is_dir():
        continue
    for name in ['news','posts','partial','review','excluded']:
        path = folder / f'{name}.csv'
        if path.exists():
            frames[(folder.name,name)] = pd.read_csv(path).fillna('')
summary = pd.DataFrame([{'source_id':s, 'dataset':k, 'rows':len(df)} for (s,k),df in frames.items()])
display(summary.pivot(index='source_id',columns='dataset',values='rows').fillna(0).astype(int))

dataset,excluded,news,partial,posts,review
source_id,,,,,
bournemouth_echo,1,47,3,0,16
caughtoffside,2,329,0,0,19
chronicle_live,8,145,3,0,17
david_ornstein,0,0,9,0,0
espn,0,22,0,0,37
fantasy_football_scout,0,88,0,0,39
football_insider,45,515,0,0,70
football_london,18,289,0,0,15
fotmob,0,10,0,0,5


## Примеры каждого источника

По два примера, где они есть. Сначала берутся статьи из основного набора, затем случаи на проверку/неполные публикации. Это проверка качества извлечения, не случайная выборка для оценки долей тем. По умолчанию показывается начало текста; для полного текста переключите SHOW_FULL_TEXT. Разметка АПЛ автоматическая и предварительная; наличие клуба в начале текста ещё не доказывает полезность для конкретного матча.

In [4]:
for _, source in audit.iterrows():
    display(HTML('<h3>' + html.escape(str(source['source'])) + '</h3>'))
    samples = []
    for kind in ['news','review','partial','posts','excluded']:
        df = frames.get((source['source_id'],kind))
        if df is not None and len(df):
            for _, row in pd.concat([df.head(1),df.tail(1)]).drop_duplicates('url').head(2-len(samples)).iterrows():
                samples.append((kind,row))
        if len(samples) == 2:
            break
    if not samples:
        display(HTML('<p>' + html.escape(str(source['status']) + ': ' + str(source['reason'])) + '</p>'))
    for kind,row in samples:
        text = str(row['text'])
        preview = text if SHOW_FULL_TEXT else text[:450] + ('…' if len(text)>450 else '')
        details = {key:row.get(key,'') for key in ['published_at','published_date_display','date_precision','date_provenance','source_tags','selection_reason','text_status','extraction_method']}
        display(HTML('<p><b>' + html.escape(str(row['title'])) + '</b> [' + kind + ']</p>' + '<p><a href="' + html.escape(str(row['url']),quote=True) + '">Оригинал</a></p>' + '<pre style="white-space:pre-wrap">' + html.escape(preview) + '</pre>'))
        display(pd.Series(details).to_frame('значение'))

,значение
published_at,2026-10-26T13:23:00+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,[]
selection_reason,title_club
text_status,full_text
extraction_method,jsonld.articleBody


,значение
published_at,2026-11-08T20:56:00+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,"[""News Story""]"
selection_reason,title_or_lead_league
text_status,full_text
extraction_method,jsonld.articleBody


,значение
published_at,2026-11-07T09:15:00+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,[]
selection_reason,title_or_lead_league
text_status,full_text
extraction_method,html:.article-body p


,значение
published_at,2026-11-08T20:30:00+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,[]
selection_reason,title_or_lead_league
text_status,full_text
extraction_method,html:.article-body p


,значение
published_at,2026-10-25T22:48:17+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,"[""Sunderland"", ""Leeds"", ""Premier League"", ""Eur..."
selection_reason,native_league_tag
text_status,full_text
extraction_method,html:article p


,значение
published_at,2026-11-08T20:56:31+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,"[""FA Cup"", ""EFL"", ""Leicester City"", ""Wigan Ath..."
selection_reason,title_or_lead_league
text_status,full_text
extraction_method,html:article p


... (40 outputs left out)

,значение
published_at,2026-11-03T11:00:39+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,"[""Inter"", ""Caffè Di Marzio""]"
selection_reason,no_verified_epl_evidence
text_status,partial
extraction_method,summary_only


,значение
published_at,2026-11-08T20:15:20+00:00
published_date_display,
date_precision,timestamp
date_provenance,jsonld.datePublished
source_tags,"[""Inter"", ""Como"", ""Como Fabregas"", ""News Calcio""]"
selection_reason,no_verified_epl_evidence
text_status,partial
extraction_method,summary_only


,значение
published_at,2026-10-30T22:25:11+00:00
published_date_display,
date_precision,timestamp
date_provenance,rss_or_atom.published
source_tags,[]
selection_reason,title_club
text_status,partial
extraction_method,summary_only


,значение
published_at,2026-11-06T17:18:08+00:00
published_date_display,
date_precision,timestamp
date_provenance,rss_or_atom.published
source_tags,[]
selection_reason,title_club
text_status,partial
extraction_method,summary_only


## Примеры проблем и точных повторов

Неполные тексты, неоднозначный отбор и исключённые страницы смотрим отдельно. Совпадения URL или хеша текста — техническая проверка дубликатов, а не группировка новостей об одном событии.

In [5]:
for kind in ['partial','review','excluded']:
    available = [df for (source,name),df in frames.items() if name==kind and len(df)]
    display(HTML('<h4>' + kind + '</h4>'))
    if available:
        examples = pd.concat(available,ignore_index=True).groupby('source_id',sort=True).head(1).head(5)
        display(examples[['source','title','published_at','url','text_status','selection_reason','exclusion_reason']])
    else:
        print('Нет сохранённых примеров')
news_frames = [df for (source,name),df in frames.items() if name=='news' and len(df)]
if news_frames:
    merged = pd.concat(news_frames,ignore_index=True)
    duplicates = merged[merged.duplicated('url',keep=False) | merged.duplicated('content_hash',keep=False)]
    display(duplicates[['source','title','url','content_hash']].head(20))
    print('Строк с точными совпадениями URL/текста:',len(duplicates))

,source,title,published_at,url,text_status,selection_reason,exclusion_reason
0,Bournemouth Echo,AFC Bournemouth's pre-Sparta conference - As i...,2026-11-04T16:01:19+00:00,https://www.bournemouthecho.co.uk/sport/266997...,partial,native_club_tag,
3,Chronicle Live,Everton send Newcastle United new four-word ta...,2026-10-31T10:00:00+00:00,https://www.chroniclelive.co.uk/sport/football...,partial,native_club_tag,
6,David Ornstein,Aston Villa to initiate legal action against J...,2026-10-30T22:25:11+00:00,https://www.nytimes.com/athletic/7739997/2026/...,partial,title_club,
15,FourFourTwo,Landing-page archive,,https://www.fourfourtwo.com/landing-page/archi...,partial,missing_or_timezone_ambiguous_publication_date,
88,GOAL,MLS Decision Day: Is Greg Vanney's time up? LA...,2026-11-08T20:02:20.871000+00:00,https://www.goal.com/en/originals/mls-decision...,partial,no_verified_epl_evidence,


,source,title,published_at,url,text_status,selection_reason,exclusion_reason
0,Bournemouth Echo,Wimborne RFC Second XV beat East Dorset Docker...,2026-11-05T05:00:00+00:00,https://www.bournemouthecho.co.uk/sport/266976...,full_text,no_verified_epl_evidence,
16,CaughtOffside,Just in: Man United risk losing one of their b...,2026-10-26T06:49:31+00:00,https://www.caughtoffside.com/2026/10/26/man-u...,full_text,possible_women_or_youth_team,
35,Chronicle Live,Lewis Miley suffers new injury scare as Newcas...,2026-10-26T10:29:25+00:00,https://www.chroniclelive.co.uk/sport/football...,full_text,possible_women_or_youth_team,
52,ESPN,"Cristiano Ronaldo goal tracker: Road to 1,000 ...",2026-10-29T21:27:00+00:00,https://www.espn.com/soccer/story/_/id/4675102...,full_text,lead_club_mention_needs_relevance_review,
89,Fantasy Football Scout,Saudi Fantasy: Round 12 team reveals,2026-10-26T00:00:00+00:00,https://www.fantasyfootballscout.co.uk/2026/10...,full_text,no_verified_epl_evidence,


,source,title,published_at,url,text_status,selection_reason,exclusion_reason
0,Bournemouth Echo,Ipswich Town v AFC Bournemouth live updates,2026-11-08T12:00:00+00:00,https://www.bournemouthecho.co.uk/sport/267095...,partial,native_club_tag,live_blog
1,CaughtOffside,Video: Morgan Rogers scores solo stunner to pu...,2026-10-31T13:55:14+00:00,https://www.caughtoffside.com/2026/10/31/morga...,full_text,native_league_tag,video
3,Chronicle Live,Newcastle u21s 3-2 Sunderland u21s reaction as...,2026-10-26T21:04:09+00:00,https://www.chroniclelive.co.uk/sport/football...,full_text,native_club_tag,live_blog
11,Football Insider,Finance Hub (Brighton),,https://www.footballinsider247.com/brighton-ho...,partial,missing_or_timezone_ambiguous_publication_date,non_article_page
56,Football London,Kane on Bayern Munich,2026-10-26T23:01:10+00:00,https://www.football.london/premier-league/bal...,full_text,native_club_tag,live_blog


,source,title,url,content_hash
340,CaughtOffside,Aston Villa prepare offer for highly-rated 22-...,https://www.caughtoffside.com/2026/11/06/aston...,8b2c1127cc432de58fd7b94b317392503cd89f9615646b...
341,CaughtOffside,Liverpool demand €70m as Real Madrid explore J...,https://www.caughtoffside.com/2026/11/06/liver...,c420815b3ac4bba9625162a46da1a99636bfe75b0cb75e...
342,CaughtOffside,Record-breaking fee expected as Arsenal & Live...,https://www.caughtoffside.com/2026/11/07/zion-...,17256bf81e1e16cc7cb98e8964e41e3cda93c2eae01018...
343,CaughtOffside,“Outstanding” striker on Newcastle United’s ra...,https://www.caughtoffside.com/2026/11/07/newca...,1bebf5a7ea46f62caa4cf7026669368f5a6b3b6340cd73...
345,CaughtOffside,“I hope…” – Andoni Iraola speaks out on Liverp...,https://www.caughtoffside.com/2026/11/07/liver...,6ed238d00e1f12d53a33f986a862a3ccd547a9f42033d2...
346,CaughtOffside,“It’s a possibility” – Fabrizio Romano deliver...,https://www.caughtoffside.com/2026/11/07/fabri...,5f3aa47da3428f174987c5cce07749deb8d3c44966d0f2...
347,CaughtOffside,Sources: Liverpool “surprised” at transfer dev...,https://www.caughtoffside.com/2026/11/07/liver...,bac2f7ffa0124e9e1ca24d0ccfeb2fe7a048a62396660c...
348,CaughtOffside,Liverpool and Barcelona competing for star who...,https://www.caughtoffside.com/2026/11/07/liver...,723659a1ab85c12f74d58f44042590dee3d17e83a24171...
349,CaughtOffside,Liverpool star open to January exit as Saudi g...,https://www.caughtoffside.com/2026/11/07/liver...,066cc54ce5884f67df83664e3c38287de23f4928f0894d...
350,CaughtOffside,Manchester United step up scouting of Ligue 1 ...,https://www.caughtoffside.com/2026/11/07/manch...,3eb8f9c435130ab60ba1fa4b7c566c27fabda78af7a9dc...


Строк с точными совпадениями URL/текста: 262


## Проверки повторного запуска и целостности

Важны отсутствие дублей по каноническому URL, сохранение старых записей и корректный перенос полного текста через CSV. Проверка не доказывает полноту сайта или постоянную доступность источника. Временные ошибки повторяются, а исходы и ограничения сохраняются в журналах.

In [6]:
checks_path = ROOT / 'research' / 'rerun_checks.json'
checks = pd.read_json(checks_path) if checks_path.exists() else pd.DataFrame()
display(checks)
for (source,kind), df in frames.items():
    assert not df['url'].duplicated().any(), (source,kind,'duplicate URL')
    if kind == 'news' and len(df):
        dates = pd.to_datetime(df['published_at'], utc=True, format='ISO8601', errors='raise')
        assert dates.notna().all()
        assert (df['text_status']=='full_text').all()
        assert (df['content_type']=='article').all()
if len(checks):
    assert checks['passed'].all(), 'Есть неуспешные проверки повторного запуска'
print('Проверки структуры сохранённых CSV пройдены.')

,source_id,script,duration_seconds,returncode,before,after,new_urls,old_urls_lost,csv_duplicate_urls,csv_text_roundtrip,passed
0,bournemouth_echo,data/bournemouth_echo/collect.py,31.796,0,50,67,17,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
1,caughtoffside,data/caughtoffside/collect.py,3.948,0,350,350,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
2,chronicle_live,data/chronicle_live/collect.py,1.717,0,173,173,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
3,david_ornstein,data/david_ornstein/collect.py,1.044,0,9,9,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
4,espn,data/espn/collect.py,0.660,0,59,59,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
5,fantasy_football_scout,data/fantasy_football_scout/collect.py,5.228,0,127,127,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
6,football_insider,data/football_insider/collect.py,0.574,0,630,630,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
7,football_london,data/football_london/collect.py,1.805,0,322,322,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
8,fotmob,data/fotmob/collect.py,1.525,0,15,15,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True
9,fourfourtwo,data/fourfourtwo/collect.py,1.900,0,264,264,0,0,"{'news': 0, 'partial': 0, 'review': 0, 'exclud...",True,True


Проверки структуры сохранённых CSV пройдены.


## Ограничения и дальнейшая работа

Полнота охвата сайтов не утверждается. `archive_limits` показывает исчерпание бюджета страниц; RSS часто содержит только недавние материалы. Отсутствие меток игрока нельзя компенсировать выдуманной принадлежностью к клубу — такие статьи остаются на проверку. Текущий справочник клубов взят из сохранённого аудита проекта от 26 октября 2026. Короткие посты журналистов сохраняются отдельно, если есть рабочий разрешённый доступ; закрытый доступ не заменяется пересказами.

Следующий этап: проверить 100–150 предварительно размеченных публикаций отдельным агентом-судьёй, затем человеком; после этого исследовать типы новостей, пересечения и покрытие клубов. Агент-судья запускается пользователем. Числовой рейтинг авторитетности и скорость относительно реального события сейчас не установлены.